### 패키지 설치 확인

In [1]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspace\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


### 추출 함수

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf

def extract_pdf_elements(path, fname):
    """
    PDF: 파일에서 이미지, 테이블, 텍스트 블록을 추출합니다.
    path: 파일 경로(추출된 이미지 파일이 저장될 위치)
    fname: 파일 이름
    """
    return partition_pdf(
        filename=path+fname,
        # 레이아웃 분석 모델로 정밀 추출 (교재에 없음! 필수 추가)
        strategy="hi_res",
        # PDF에서 이미지를 추출
        extract_images_in_pdf=True,
        # 테이블 구조를 추론
        infer_table_structure=True,
        # 타이틀을 기준으로 텍스트를 블록으로 분할
        chunking_strategy="by_title",
        # 최대 4000자로 텍스트 블록을 제한
        max_characters=4000,
        # 3800자 이후에 새로운 블록 생성
        new_after_n_chars=3800,
        # 2000자 이하의 텍스트는 결합
        combine_text_under_n_char=2000,
        # 이미지가 저장될 경로 설정
        # image_output_dir_path=path,
        extract_image_block_output_dir=os.path.join(path, "figures"),
    )

### 추출한 요소를 종류별로 분류

In [ ]:
def categorize_elements(raw_pdf_elements):
    """
    PDF에서 추출한 요소들을 테이블과 텍스트로 분류합니다.
    raw_pdf_elements: unstructured.documents.elements 리스트
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))
    return tables, texts

### 경로 설정 및 실행

In [ ]:
import os

# 현재 작업 디렉터리 경로를 반환
current_directory = os.getcwd()

# fame: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "invest.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "data/")

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

#실행하면 2~3분 소요, 멈춘 것이 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
tables, texts = categorize_elements(raw_pdf_elements)

### 텍스트 분할

In [ ]:
# chunk_size=2000: 각 텍스트 블록의 최대 크기
# chunk_overlap=200: 청크 간 200자의 겹침을 허용해 중요한 정보를 잃지 않도록 함
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 2000,
    chunk_overlap = 200,
)

# 텍스트 리스트를 하나의 긴 문자열로 결합한 뒤 2000자 단위로 분할
joined_texts = " ".join(texts)
texts_2k_token = text_splitter.split_text(joined_texts)

print("분할 후 블록 수:", len(texts_2k_token))
print("원본 블록 수:", len(texts))
print(texts[0])
print(type(texts_2k_token))

### API 키 불러오기

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. 한번 더 확인 부탁드립니다.")

### 요약 체인 구성

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama.llms import OllamaLLM

def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    텍스트 및 표 데이터를 요약하며 검색에 활용할 수 있는 요약본을 생성합니다.
    texts: 텍스트 리스트
    tables: 표 리스트
    summarize_texts: 텍스트 요약 여부 (기본값: False)
    """

    # prompt 한국어 버전
    prompt_text_kor ="""
    당신은 표와 텍스트를 요약하며 검색에 활용할 수 있도록 돕는 도우미 입니다.
    이 요약본들은 임베딩되어 원본 텍스트나 표 요소를 검색하는 데 사용될 것 입니다.
    주어진 표나 텍스트의 내용을 검색에 최적화된 간결한 요약으로 작성해주세요.
    요약할 표 또는 텍스트: {element}
    """

    prompt = ChatPromptTemplate.from_template(template=prompt_text_kor)
    
    # 모델: GPT-4o-mini or Llama3.1 모델 사용
    # llamaModel = OllamaLLM(model="llama3.1:8b")
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

    # 요약 체인 = 프롬프트 + LLM + 출력 파서
    summarize_chain = {"element": lambda x:x} | prompt | model | StrOutputParser()
    text_summaries = []
    table_summaries = []

    # 텍스트 요약을 활성화한 경우
    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency":5})
    #텍스트 요약 비활성화
    elif texts:
        text_summaries = texts

    # 테이블 데이터 요약
    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency":5})
    return text_summaries, table_summaries

text_summaries, table_summaries = generate_text_summaries(
    texts_2k_token, tables, summarize_texts=True
)
print(text_summaries)
print(table_summaries)

"""
max_concurrency=5는 한번에 5개씩 요약하라는 뜻, 숫자를 올리면 더 빨라지지만 API 요청 제한 위험
"""

### Base64 인코딩

In [ ]:
import base64
import os
import requests
import json
from langchain_core.messages import HumanMessage

def encode_image(image_path):
    """이미지를 base64 문자열로 변환"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")